# Project Big Data - Notebook Group 22; Amazon Books

## Datasets Import and Setup
In the following block we import the datasets and prepare them for further usage.

In [171]:
import warnings

warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import sklearn as skl
import pycountry
import plotly.express as px

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CUR_DIR = os.getcwd()
BOOKS_DATA_PATH = os.path.join(CUR_DIR, "books_data/books.csv")
RATINGS_DATA_PATH = os.path.join(CUR_DIR, "books_data/ratings.csv")
USERS_DATA_PATH = os.path.join(CUR_DIR, "books_data/users.csv")

df_books = pd.read_csv(BOOKS_DATA_PATH, encoding = 'latin-1', sep = ';', quotechar='"', escapechar="\\").drop(columns = ['Image-URL-S', 'Image-URL-M', 'Image-URL-L'])
df_ratings = pd.read_csv(RATINGS_DATA_PATH, encoding = 'latin-1', sep = ';', quotechar='"')
df_users = pd.read_csv(USERS_DATA_PATH, encoding = 'latin-1', sep = ';', quotechar='"', escapechar="\\", index_col=0)

print(f"Books data    loaded {len(df_books):,} rows, {df_books.shape[1]} columns.")
print(f"Ratings data  loaded {len(df_ratings):,} rows, {df_ratings.shape[1]} columns.")
print(f"Users data    loaded {len(df_users):,} rows, {df_users.shape[1]} columns.")

df_active_countries = df_users['Location'].str.split(',').str[-1].str.strip()
unique_active_countries = df_active_countries.unique()

Books data    loaded 271,379 rows, 5 columns.
Ratings data  loaded 1,149,780 rows, 3 columns.
Users data    loaded 278,858 rows, 2 columns.


In the block below we define a list of countries that will be used further. This is a step of cleaning customers data, so we are left only with realistic and standardized countries where customers originate from. A "realistic and standardized country" is a relative term; primarily it is interpreted as the english name or abbreviation of a country's name.

We use a two-step validation method where we first obtain a list of mutual countries and teritories, present in the dataset and in the python library pycountry, storing official names of most countries in the world (METHOD 1). Then (in METHOD 2) we check the data from countries with more than 50 customers, and compare if we are missing any country from the "mutual" list. We manually add those missing countries to the "mutual" list and then we convert the ones that have two names in the dataframe using a map (example is 'usa' that was manually added due to a big proportion being under that name and 'united states'). 

At the end we preserved about 98% of the users from the original set, organizing them in each country accordingly. The other 2% is removed due to the unstandardized nature of the addresses. Note, there are two countries with more than 50 users, that we decided not to include: "españa" due to having about 65 customers, while the majority has correctly used the standard "spain", and "yugoslavia" due to having about 180 customers, however the country no longer existing.

In [ ]:
"""
METHOD 1
"""
countries = []
for country in pycountry.countries:
    countries.append(country.name.lower())

mutual = []
for mutual_country in countries:
    if mutual_country in unique_active_countries:
        mutual.append(mutual_country)

#ADD MANUALLY countries not in mutual
mutual += "usa", "russia", "u.a.e", "turkey", "españa", "iran", "vietnam", "taiwan", "syria", "venezuela", "south korea", "czech republic"
 
total_users_method1 = 0
for country in mutual:
    total_users_method1 += df_active_countries[df_active_countries == country].count()
print(f"Percentage of users kept after cleaning;            METHOD 1: {round(total_users_method1/len(df_users)*100, 2)}")

"""
METHOD 2
this is an additional validation step of the mutual list
"""
countries_counts = df_active_countries.value_counts()
counts_list = countries_counts[countries_counts>50].index.tolist()
counts_list.remove("")

total_users_method2 = 0
for country in counts_list:
    total_users_method2 += df_active_countries[df_active_countries == country].count()
print(f"Percentage of users kept after cleaning; only using METHOD 2: {round(total_users_method2/len(df_users)*100, 2)}")
print(f"Countries, having more than 50 customers, excluded from the countries list are: {set(counts_list)- set(mutual)}")

df_users['Country'] = df_users['Location'].str.split(',').str[-1].str.strip().str.lower()
df_users['Country'] = df_users['Country'].apply(lambda x: x if x in mutual else "unknown")

countries_map = {'usa' : 'united states',
                 'russian federation' : 'russia',
                 'türkiye' : 'turkey',
                 'u.a.e' : 'united arab emirates',
                 'españa' : 'spain',
                 'hong kong': 'china',
            }
df_users['Country'] = df_users['Country'].replace(countries_map)

df_users = df_users.drop(columns = ['Location'])

Percentage of users kept after cleaning;            METHOD 1: 97.94
Percentage of users kept after cleaning; only using METHOD 2: 97.38
Countries, having more than 50 customers, excluded from the countries list are: {'yugoslavia'}


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company


# Countries Metrics
Below we present basic metrics per country, such as the average scores across countries, and the review frequency (as well as the sample size for each country). 

We remind again that 0-scores represent the purchase of a book and leaving a no score, whereas ratings 1-10 show a customers appreciation of the book.

It is important to notice that all countries are included, disregarding the sample size. This leads to countries inaccurate mean ratings due to limited number of observations within the country. We later investigate countries with bigger sample sizes, so we atempt to extract more informed analytics.

Sample Size graph is splitted into two Reviews Frequency graphs, which are mutually exclusive.

In [173]:
df_ratings_users = df_ratings.merge(df_users, on = 'User-ID').merge(df_books, on = 'ISBN')

df_no_rating = df_ratings_users[df_ratings_users["Book-Rating"] == 0]
df_rated = df_ratings_users[df_ratings_users["Book-Rating"] > 0]

df_mean_ratings = df_rated.groupby('Country')['Book-Rating'].mean().reset_index()
df_samples_sizes_countries = df_ratings_users.groupby('Country')['Book-Rating'].count().reset_index()

df_rating_frequency_leaving_reviews = df_rated.groupby('Country')['Book-Rating'].count().reset_index()
df_rating_frequency_not_leaving_reviews = df_no_rating.groupby('Country')['Book-Rating'].count().reset_index()

fig = px.choropleth(
    df_mean_ratings,
    locations='Country',
    locationmode='country names',
    color='Book-Rating',
    title='Average Book Rating per Country',
    hover_data='Book-Rating',
    color_continuous_scale='bluyl'
)
fig.show()

fig = px.choropleth(
    df_samples_sizes_countries,
    locations='Country',
    locationmode='country names',
    color='Book-Rating',
    title='Sample Sizes per Country',
    hover_data='Book-Rating',
    color_continuous_scale='bluyl'
)
fig.show()

fig = px.choropleth(
    df_rating_frequency_leaving_reviews,
    locations='Country',
    locationmode='country names',
    color='Book-Rating',
    title='Reviews Frequency per Country (Leaving Reviews)',
    hover_data='Book-Rating',
    color_continuous_scale='bluyl'
)
fig.show()

fig = px.choropleth(
    df_rating_frequency_not_leaving_reviews,
    locations='Country',
    locationmode='country names',
    color='Book-Rating',
    title='Reviews Frequency per Country (Not Leaving Reviews)',
    hover_data='Book-Rating',
    color_continuous_scale='bluyl'
)
fig.show()

# Metrics for Countries with more than 100 reviews, having left a review

We decided that 100 reviews might be a good starting point, since under Central Limit Theorem, samples with above 30 observations allow the sample mean to be treated as approximately normally distributed, allowing for more reliable estimate of the means.
In the first graph we have the mean book ratings per country, where we can see that the majority falls between 7 and 9, with average mean of all of about 7.6.

In the second graph we have computed the standard deviation of each country's rating, where we see that the average st. dev. across all is about 1.8. Considering this information, we get that for a randomly picked review we can say that the score might be roughly between 6 and 9.5 (mean ± std). As one can imagine, on the scale 1-10, this suggests that happy customers may tend to leave reviews more often in comparison with unhappy ones, making it a reasonable hypothesis for investigating a positive bias.

In [174]:
df_std_ratings = df_rated.groupby('Country')['Book-Rating'].std().reset_index()

df_ratings_frequency_morethan_100 = df_rating_frequency_leaving_reviews[df_rating_frequency_leaving_reviews['Book-Rating']>=100]['Country']

df_mean_ratings_morethan_100 = df_mean_ratings[df_mean_ratings['Country'].isin(df_ratings_frequency_morethan_100)]
df_std_ratings_morethan_100 = df_std_ratings[df_std_ratings['Country'].isin(df_ratings_frequency_morethan_100)]

average_mean_countries = df_mean_ratings_morethan_100['Book-Rating'].mean()
average_std_countries = df_std_ratings_morethan_100['Book-Rating'].mean()

print(f"Average Mean across countries, having more than 100 reviews, is {round(average_mean_countries, 2)}, Average Standard Deviation is {round(average_std_countries, 2)}.")

fig = px.choropleth(
    df_mean_ratings_morethan_100,
    locations='Country',
    locationmode='country names',
    color='Book-Rating',
    title='Average Book Rating per Country for Countries with more than 100 reviews',
    hover_data='Book-Rating',
    color_continuous_scale='bluyl'
)
fig.show()

fig = px.choropleth(
    df_std_ratings_morethan_100,
    locations='Country',
    locationmode='country names',
    color='Book-Rating',
    title='Standard Deviation Book Rating per Country for Countries with more than 100 reviews',
    hover_data='Book-Rating',
    color_continuous_scale='bluyl'
)
fig.show()

Average Mean across countries, having more than 100 reviews, is 7.63, Average Standard Deviation is 1.74.


# Average rate per age group

In [175]:
df_age_mean_rating = df_rated.groupby("Age")["Book-Rating"].mean().reset_index()

df_age_mean_rating["Age_Group"] = pd.cut(df_age_mean_rating["Age"], bins=[0, 15, 20, 30, 40, 50, 60, 70, 80, 90, 100, 300], 
                                labels=["0-15", "15-20", "20-30", "30-40", "40-50", "50-60", "60-70", "70-80", "80-90", "90-100", "100+"])

df_age_mean_rating = df_age_mean_rating.groupby("Age_Group")["Book-Rating"].mean()
df_age_mean_rating

Age_Group
0-15      7.767803
15-20     7.644309
20-30     7.748220
30-40     7.648359
40-50     7.764839
50-60     7.759150
60-70     7.667118
70-80     7.519005
80-90     7.477843
90-100    7.586449
100+      7.059537
Name: Book-Rating, dtype: float64